In [33]:
import org.apache.spark.sql.functions.{explode, split, col}
import spark.implicits._

In [12]:
// Creating the Spark Context
val spark = SparkSession.builder().appName("Movie Ratings DataSet").master("local[*]").getOrCreate()
val sparkConext = spark.sparkContext

spark = org.apache.spark.sql.SparkSession@61f18f4f
sparkConext = org.apache.spark.SparkContext@31a9b803


org.apache.spark.SparkContext@31a9b803

In [14]:
// Creating the Spark Context
val spark = SparkSession.builder().appName("Movie Ratings DataSet").master("local[*]").getOrCreate()
val sparkConext = spark.sparkContext

spark = org.apache.spark.sql.SparkSession@61f18f4f
sparkConext = org.apache.spark.SparkContext@31a9b803


org.apache.spark.SparkContext@31a9b803

In [15]:
val moviesDffPath = "gs://artifacts_spark_jobs/movie.csv"
val ratingDffPath = "gs://artifacts_spark_jobs/rating.csv"


val moviesDff = spark.read.format("csv").option("header","true").option("inferSchema","true").load(moviesDffPath)
val ratingsDff = spark.read.format("csv").option("header","true").option("inferSchema","true").load(ratingDffPath)

// reading the csv files from source

moviesDffPath = gs://artifacts_spark_jobs/movie.csv
ratingDffPath = gs://artifacts_spark_jobs/rating.csv
moviesDff = [movieId: int, title: string ... 1 more field]
ratingsDff = [userId: int, movieId: int ... 2 more fields]


[userId: int, movieId: int ... 2 more fields]

In [5]:
val explodedGenresDF = moviesDff.withColumn("genres", explode(split(col("genres"),"\\|"))).select("movieId","title","genres")

explodedGenresDF = [movieId: int, title: string ... 1 more field]


[movieId: int, title: string ... 1 more field]

In [26]:
val moviesRDD = explodedGenresDF.rdd

def standardizeGenre(genre: String): String ={
  genre match{
    case "Sci-Fi" => "Science Fiction"
    case "Film-Noir" => "Black and White"
    case "(no genres listed)" => "Others"
    case _ => genre 
  }
}


val standarizedData = moviesRDD.map(eachEle=>{
   val movieId = eachEle.getAs[Int]("movieId")
   val movieName = eachEle.getAs[String]("title")
   val genre = eachEle.getAs[String]("genres")

   val standardizedGenre = standardizeGenre(genre)
  
   (movieId, (movieName, standardizedGenre)) 
})

moviesRDD = MapPartitionsRDD[25] at rdd at <console>:25
standarizedData = MapPartitionsRDD[73] at map at <console>:41


standardizeGenre: (genre: String)String


MapPartitionsRDD[73] at map at <console>:41

In [27]:
val ratingsRDD = ratingsDff.rdd.map(row => {
  val movieId = row.getAs[Int]("movieId")
  val rating = row.getAs[Double]("rating")
  (movieId, rating)  // Restructured for join
})

ratingsRDD = MapPartitionsRDD[74] at map at <console>:27


MapPartitionsRDD[74] at map at <console>:27

In [28]:
val moviesWithRatings = ratingsRDD.join(standarizedData)

moviesWithRatings = MapPartitionsRDD[77] at join at <console>:28


MapPartitionsRDD[77] at join at <console>:28

In [ ]:
val genreAverageRatings = moviesWithRatings
  .map { case (_, (rating, (_, genre))) => 
    (genre, (rating, 1))
  }
  .reduceByKey { case ((sum1, count1), (sum2, count2)) => 
    (sum1 + sum2, count1 + count2)  
  }
  .mapValues { case (sum, count) => 
    sum / count.toDouble  
  }

In [ ]:
/user/hdfs/CaseStudies

In [34]:
val genreAverageDF = genreAverageRatings.toDF("genre", "average_rating")

genreAverageDF = [genre: string, average_rating: double]


[genre: string, average_rating: double]

In [35]:
val hdfsPath = "hdfs:/user/hdfs/CaseStudies"

hdfsPath = hdfs:/user/hdfs/CaseStudies


hdfs:/user/hdfs/CaseStudies

In [36]:
genreAverageDF.write.mode("overwrite").parquet(hdfsPath)

In [37]:
val checkDF = spark.read.parquet(hdfsPath)
checkDF.show()

+---------------+------------------+
|          genre|    average_rating|
+---------------+------------------+
|Black and White|  3.96538126070082|
|         Others|3.0069252077562325|
|       Children|3.4081137685270444|
|        Mystery| 3.663508921312903|
|        Romance| 3.541802581902903|
|        Western|3.5704980246109406|
|        Musical| 3.558090628821412|
|         Horror|3.2772238097518307|
|       Thriller|  3.50711121809216|
|      Adventure|3.5018926565473865|
|            War|3.8095307347384844|
|Science Fiction|3.4367726714455005|
|          Crime|3.6745276025631113|
|      Animation|3.6174939235897994|
|    Documentary|3.7397176834178865|
|          Drama|3.6742955093068264|
|        Fantasy|3.5059453358738244|
|         Action|  3.44386376493354|
|           IMAX| 3.655945983272606|
|         Comedy|3.4260113054324886|
+---------------+------------------+



checkDF = [genre: string, average_rating: double]


[genre: string, average_rating: double]